# TSL quickstart

Generate a few TSL21 or TSL60 images and measure word recall, NSL, and GSL. Run Jupyter from the repository root after [installing the dependencies](README.md#install) and [downloading the CSVs](README.md#get-the-data). The default batch is the first three TSL21 requests.


## 1. Start the services

The default settings use LongCat for both target and matched-base images (`BASE_SOURCE = "model"`) and OpenRouter for GSL. Start LongCat from the repository root:

```bash
CUDA_VISIBLE_DEVICES=0,1 python -m uvicorn backends.longcat:app --host 127.0.0.1 --port 8004
```

Set `OPENROUTER_API_KEY` in the repository `.env` or your environment. OCR and CLIP load locally through `rtsl-scorer`. For another target model, use its [backend command](README.md#evaluate-an-image-generator) and set `MODEL` and `PORT` below.

**Optional FLUX.2-dev bases:** Set `BASE_SOURCE = "flux_2_dev"` to use images from `outputs/bases/`. If they are not already present, start FLUX.2-dev and prepare them once:

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6 python -m uvicorn backends.flux_2_dev:app --host 127.0.0.1 --port 8001
# In another terminal, from the repository root:
python scripts/evaluation/prepare_nsl_bases.py --output outputs/bases \
  --server http://127.0.0.1:8001/generate
```

The notebook reads these FLUX bases from disk. If the target is a different model, FLUX can be stopped after preparation; keep the target backend running until image generation finishes.

**Optional local VQA:** Install the [Qwen VQA environment](backends/README-vqa.md) and set `VQA_SOURCE = "local"`. After generating the images in Section 3, stop the target backend if it shares GPUs with VQA, then start the VLM before running the scoring cells. The example uses eight GPUs; adjust the GPU allocation for your machine.

```bash
conda activate qwen35-vllm
CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7 vllm serve Qwen/Qwen3-VL-32B-Instruct \
  --tensor-parallel-size 8 --served-model-name qwen3-vl-32b \
  --max-model-len 8192 --enforce-eager --max-num-seqs 1 --port 18700
```


In [ ]:
MODEL = "longcat"  # flux_2_dev, qwen_image, z_image_turbo, or longcat
PORT = 8004  # target backend port
VERBOSE = True
DATASET = "tsl21"  # "tsl60" or "tsl21"
BASE_SOURCE = "model"  # "model": target backend; "flux_2_dev": outputs/bases

BEGIN_IDX = 0  # set to i * 3 for group i
OFFSET = 3

VQA_SOURCE = "openrouter"  # "openrouter" or "local"
OPENROUTER_MODEL = "qwen/qwen3-vl-32b-instruct"
LOCAL_VQA_MODEL = "qwen3-vl-32b"
LOCAL_VQA_URL = "http://127.0.0.1:18700/v1"

from pathlib import Path
from urllib.request import urlopen
import json
import sys

ROOT = Path.cwd()
assert (ROOT / "backends").is_dir(), "Start Jupyter from the repository root"
sys.path.insert(0, str(ROOT))
from scripts.evaluation.text_rendering_data import MODELS

assert MODEL in MODELS
assert BASE_SOURCE in ("model", "flux_2_dev")

SERVER_URL = f"http://127.0.0.1:{PORT}"
with urlopen(f"{SERVER_URL}/health", timeout=10) as response:
    print(f"{MODEL} backend:", json.load(response))

## 2. Load requests


In [ ]:
import pandas as pd
from IPython.display import display

requests = pd.read_csv(ROOT / f"data/{DATASET}/{DATASET.upper()}_requests.csv")
display(requests.iloc[BEGIN_IDX:BEGIN_IDX + OFFSET][
    ["id", "goal_id", "dimension_id", "rendered_text", "prompt", "main_prompt_id", "goal_description"]
])

## 3. Generate images

Generate the selected requests and their matched bases using the source chosen above. Existing images are reused.

In [ ]:
OUTPUT = ROOT / "outputs/quickstart" / MODEL / "random"
BASE_DIR = OUTPUT / ("flux_2_dev_bases" if BASE_SOURCE == "flux_2_dev" else "bases")
MASKED_BASE_DIR = OUTPUT / ("masked_flux_2_dev_bases" if BASE_SOURCE == "flux_2_dev" else "masked_bases")
SHARED_BASE_DIR = ROOT / "outputs/bases"

import base64
import io
from shutil import copy2
from urllib.request import Request
from PIL import Image

def generate_png(prompt, path):
    request = Request(
        f"{SERVER_URL}/generate",
        data=json.dumps({"prompt": prompt, "width": 1024, "height": 1024}).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(request, timeout=900) as response:
        item = json.load(response)["data"][0]
    with Image.open(io.BytesIO(base64.b64decode(item["b64_json"]))) as image:
        temporary = path.with_suffix(".tmp")
        image.convert("RGB").save(temporary, format="PNG")
    temporary.replace(path)

def prepare_image(path, prompt, source=None):
    if path.exists():
        with Image.open(path) as image:
            image.verify()
        action = "reused"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        if source is None:
            generate_png(prompt, path)
        else:
            assert source.exists(), source
            copy2(source, path)
        action = "generated" if source is None else "copied"
    if VERBOSE:
        print(f"{action}: {path.relative_to(ROOT)}")

assert 0 <= BEGIN_IDX < BEGIN_IDX + OFFSET <= len(requests)
base_prompts = pd.read_csv(ROOT / "data/tsl60/TSL60_base_prompts.csv")
selected = list(requests.iloc[BEGIN_IDX:BEGIN_IDX + OFFSET].itertuples(index=False))
for row in selected:
    prepare_image(OUTPUT / DATASET / "images" / f"{row.id}.png", row.prompt)
    bases = base_prompts[base_prompts.main_prompt_id == row.main_prompt_id].sort_values("base_slot")
    assert len(bases) == 2, row.main_prompt_id
    for base in bases.itertuples(index=False):
        base_id = f"{base.main_prompt_id}__{base.base_slot}.png"
        source = SHARED_BASE_DIR / base_id if BASE_SOURCE == "flux_2_dev" else None
        prepare_image(BASE_DIR / base_id, base.prompt, source)

In [ ]:
display_id = 0
select_row = selected[display_id]
select_image = OUTPUT / DATASET / "images" / f"{select_row.id}.png"
with Image.open(select_image) as image:
    display(image.copy().resize((512, 512)))

## 4. Measure OCR and NSL

`rtsl-scorer` masks rendered text and compares each image with two matched bases using CLIP.


In [ ]:
from dataclasses import asdict
from rtsl_scorer import LocalCLIP, RTSLScorer

CLIP_MODEL = "openai/clip-vit-base-patch32"
assert VQA_SOURCE in ("openrouter", "local")
if VQA_SOURCE == "openrouter":
    scorer = RTSLScorer.from_openrouter(model=OPENROUTER_MODEL, env_file=ROOT / ".env")
else:
    scorer = RTSLScorer.from_vllm(model=LOCAL_VQA_MODEL, base_url=LOCAL_VQA_URL)
scorer.clip = LocalCLIP(model=CLIP_MODEL, device="cpu")

def signature(path):
    stat = path.stat()
    return [stat.st_size, stat.st_mtime_ns]

def cached(stage, image, settings, compute, output=None):
    relative = image.relative_to(OUTPUT)
    path = OUTPUT / "cache" / stage / relative.with_suffix(".json")
    saved = json.loads(path.read_text()) if path.exists() else {}
    entries = saved.get("entries", []) if saved.get("image_signature") == signature(image) else []
    for entry in entries:
        if entry["settings"] == settings and (
            output is None or output.exists() and entry.get("output_signature") == signature(output)
        ):
            if VERBOSE:
                print(f"cached {stage}: {relative}")
            return entry["result"]

    result = compute()
    entry = {"settings": settings, "result": result}
    if output is not None:
        entry["output_signature"] = signature(output)
    entries = [old for old in entries if old["settings"] != settings] + [entry]
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps({"image_signature": signature(image), "entries": entries},
                                    ensure_ascii=False, indent=2))
    temporary.replace(path)
    if VERBOSE:
        print(f"computed {stage}: {relative}")
    return result

def cached_mask(image, text, output, require_clear=True):
    return cached(
        "mask", image, ["rtsl-scorer", "PP-OCRv5", text, require_clear],
        lambda: asdict(scorer.mask_image(
            image, rendered_text=text, output=output, require_clear=require_clear
        )),
        output=output,
    )

In [ ]:
nsl_rows = []
mask_evidence = {}
for row in selected:
    image = OUTPUT / DATASET / "images" / f"{row.id}.png"
    masked = OUTPUT / DATASET / "masked" / f"{row.id}.png"
    mask = cached_mask(image, row.rendered_text, masked, require_clear=False)
    mask_evidence[row.id] = mask

    bases = base_prompts[base_prompts.main_prompt_id == row.main_prompt_id].sort_values("base_slot")
    masked_bases = []
    for base in bases.itertuples(index=False):
        name = f"{base.main_prompt_id}__{base.base_slot}.png"
        base_mask = MASKED_BASE_DIR / name
        cached_mask(BASE_DIR / name, base.rendered_text, base_mask)
        masked_bases.append(base_mask)

    nsl = cached(
        "nsl", masked,
        [row.rendered_text, CLIP_MODEL,
         [[str(path.relative_to(OUTPUT)), signature(path)] for path in masked_bases]],
        lambda: asdict(scorer.score_nsl_masked(
            masked, rendered_text=row.rendered_text, bases=masked_bases,
            word_recall=mask["word_recall"],
        )),
    )
    nsl_rows.append({
        "dataset": DATASET, "id": row.id, "goal_id": row.goal_id, "base_source": BASE_SOURCE,
        "dimension_id": row.dimension_id, "word_recall": mask["word_recall"],
        "masked_word_recall": mask["masked_word_recall"],
        "mask_valid": mask["masked_word_recall"] == 0,
        "mask_box_count": len(mask["boxes"]), "mask_strategy": mask["strategy"],
        "direct_tsl": nsl["direct_tsl"], "base_tsl": nsl["base_tsl"], "nsl": nsl["nsl"],
    })

scores = pd.DataFrame(nsl_rows)
display(scores)

In [ ]:
masked_preview = OUTPUT / DATASET / "masked" / f"{select_row.id}.png"
with Image.open(masked_preview) as image:
    display(image.copy().resize((512, 512)))

## 5. Measure GSL

The selected VQA model answers two questions about each masked image. GSL is the mean of their scores.


In [ ]:
import matplotlib.pyplot as plt

scores["gsl"] = float("nan")
scores["gsl_success"] = False
scores["gsl_details"] = ""
for index, row in enumerate(selected):
    masked = OUTPUT / DATASET / "masked" / f"{row.id}.png"
    result = cached(
        "gsl", masked,
        [VQA_SOURCE, scorer.backend.model, row.target_dimension, row.goal_description],
        lambda: asdict(scorer.score_masked(
            masked, target_dimension=row.target_dimension, goal_description=row.goal_description
        )),
    )
    scores.at[index, "gsl"] = result["score"]
    scores.at[index, "gsl_success"] = result["score"] > 0.5
    scores.at[index, "gsl_details"] = json.dumps(result["questions"], ensure_ascii=False)

display(scores[["dataset", "id", "word_recall", "nsl", "gsl", "gsl_success", "mask_valid"]])
axes = scores[["direct_tsl", "base_tsl", "nsl", "gsl"]].plot.bar(
    subplots=True, layout=(1, 4), figsize=(14, 3), legend=False
)
for axis, title in zip(axes.flat, ("Direct TSL", "Base TSL", "NSL", "GSL")):
    axis.set_title(title)
    axis.set_xticklabels([row.id for row in selected], rotation=45, ha="right")
axes.flat[-1].set_ylim(0, 1.08)
plt.tight_layout()
plt.show()

## 6. Inspect results

Each image is followed by its scores, word-of-interest recall, and two GSL answers.

In [ ]:
from scripts.evaluation.sl_utils import word_recall

woi_table = pd.read_csv(ROOT / "data/tsl60/TSL60_word_of_interest.csv").set_index("id")
for index, row in enumerate(selected):
    expected = woi_table.at[row.id, "word_of_interest"]
    ocr = mask_evidence[row.id]["original_ocr"]
    recall = word_recall(expected, ocr["recognized_texts"])["word_recall"]
    scores.at[index, "word_of_interest"] = expected
    scores.at[index, "woi_word_recall"] = recall

    with Image.open(OUTPUT / DATASET / "images" / f"{row.id}.png") as image:
        display(image.copy().resize((512, 512)))
    print(f"{row.id} | rendered text: {row.rendered_text}")
    print(f"Word recall: {scores.at[index, 'word_recall']:.3f} | "
          f"NSL: {scores.at[index, 'nsl']:.3f} | GSL: {scores.at[index, 'gsl']:.3f}")
    print(f"Word of interest: {expected} | recall: {recall:.3f}")
    for number, detail in enumerate(json.loads(scores.at[index, "gsl_details"]), 1):
        print(f"Q{number} ({detail['score']:.3f}): {detail['question']}")
        print(f"Rationale: {detail['rationale']}")
